# Divisão Temporal: Treino e Out-of-Time (OOT)

Neste notebook, vamos realizar a divisão temporal dos dados de transações em:
- **Dataset de Treino (df_treino)**: Usado para treinar o modelo de detecção de lavagem de dinheiro
- **Dataset Out-of-Time (df_oot)**: Usado para validação temporal e cálculo de métricas

## 1. Importação de Bibliotecas

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


## 2. Caminho do Arquivo

In [2]:
file_path = r'C:\Users\win\Desktop\Projetos\TCC_v2\anti_money_laundering\money_laundering\data\interim\HI-Large_sampled.csv'

## 3. Leitura dos Dados

In [3]:
# Leitura do arquivo CSV
print(f"Carregando dados de: {file_path}")
df = pd.read_csv(file_path)

# Converter a coluna Timestamp para datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

print(f"\nDados carregados com sucesso!")
print(f"Total de registros: {len(df):,}")
print(f"\nPeríodo dos dados:")
print(f"Data inicial: {df['Timestamp'].min()}")
print(f"Data final: {df['Timestamp'].max()}")
print(f"\nPrimeiras linhas:")
df.head()

Carregando dados de: C:\Users\win\Desktop\Projetos\TCC_v2\anti_money_laundering\money_laundering\data\interim\HI-Large_sampled.csv

Dados carregados com sucesso!
Total de registros: 17,969,102

Período dos dados:
Data inicial: 2022-08-01 00:00:00
Data final: 2023-01-09 12:27:00

Primeiras linhas:


,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,From Bank Name,From Entity ID,From Entity Name,To Bank Name,To Entity ID,To Entity Name
0,2022-08-31 07:29:00,0,8002FEF10,0,800193630,4126.49,US Dollar,4126.49,US Dollar,ACH,0,Hearthstone Bancorp,2AA0693BB30,Partnership #251,Hearthstone Bancorp,2AA02EB81E0,Corporation #263
1,2022-10-28 17:08:00,3513,8002FE280,0,800193630,508.61,US Dollar,508.61,US Dollar,Cheque,0,Bank of Sacramento,2AA06AA4780,Sole Proprietorship #5311,Hearthstone Bancorp,2AA02EB81E0,Corporation #263
2,2022-11-04 10:22:00,3513,8002FE280,0,800193630,508.61,US Dollar,508.61,US Dollar,Cheque,0,Bank of Sacramento,2AA06AA4780,Sole Proprietorship #5311,Hearthstone Bancorp,2AA02EB81E0,Corporation #263
3,2022-10-07 11:02:00,3513,8002FE280,0,800193630,508.61,US Dollar,508.61,US Dollar,Cheque,0,Bank of Sacramento,2AA06AA4780,Sole Proprietorship #5311,Hearthstone Bancorp,2AA02EB81E0,Corporation #263
4,2022-08-01 09:05:00,0,800231A90,0,800231A90,22.29,US Dollar,22.29,US Dollar,Reinvestment,0,Hearthstone Bancorp,2AA02EB7FA0,Sole Proprietorship #194,Hearthstone Bancorp,2AA02EB7FA0,Sole Proprietorship #194


## 4. Análise da Distribuição Temporal

In [4]:
# Análise da distribuição por mês
df['Year_Month'] = df['Timestamp'].dt.to_period('M')
distribuicao_mensal = df.groupby('Year_Month').agg({
    'Timestamp': 'count',
    'Is Laundering': 'sum'
}).rename(columns={'Timestamp': 'Total_Transacoes', 'Is Laundering': 'Total_Lavagem'})

distribuicao_mensal['Percentual_Lavagem'] = (
    distribuicao_mensal['Total_Lavagem'] / distribuicao_mensal['Total_Transacoes'] * 100
).round(2)

print("Distribuição mensal dos dados:")
print(distribuicao_mensal)

Distribuição mensal dos dados:
            Total_Transacoes  Total_Lavagem  Percentual_Lavagem
Year_Month                                                     
2022-08              5945551           4670                0.08
2022-09              5618470           6752                0.12
2022-10              5476987           7521                0.14
2022-11               927460           3359                0.36
2022-12                  609            373               61.25
2023-01                   25             13               52.00


## 5. Divisão Temporal: Treino e Out-of-Time

Vamos dividir os dados utilizando uma proporção de **80% para treino** e **20% para out-of-time**. 
A divisão será feita com base na ordem temporal das transações.

### ⚠️ GAP de Segurança (Anti-Leakage)

Para evitar Data Leakage causado por atraso na marcação de fraude (chargeback delay), 
implementamos um **gap de 7 dias** entre o fim do treino e o início do OOT.

**Exemplo de problema sem gap:**
- Transação fraudulenta em 31/12/2023 às 23:59h
- Chargeback reportado apenas em 05/01/2024
- Se OOT começar em 01/01/2024, o modelo "vê o futuro" (label delay leakage)

**Solução:** Gap temporal de 7 dias entre treino e OOT.

In [5]:
# Ordenar os dados por timestamp
df_sorted = df.sort_values('Timestamp').reset_index(drop=True)

# Definir o ponto de corte temporal (80% para treino, 20% para OOT)
split_ratio = 0.8
split_index = int(len(df_sorted) * split_ratio)

# GAP de segurança para evitar label leakage (chargeback delay)
GAP_DAYS = 7  # 7 dias de segurança

# Obter a data de corte inicial
data_corte_inicial = df_sorted.iloc[split_index]['Timestamp']

# Aplicar o gap: OOT começa GAP_DAYS após a data de corte
data_corte_com_gap = data_corte_inicial + pd.Timedelta(days=GAP_DAYS)

# Realizar a divisão COM GAP
df_treino = df_sorted[df_sorted['Timestamp'] < data_corte_inicial].copy()
df_gap = df_sorted[
    (df_sorted['Timestamp'] >= data_corte_inicial) & 
    (df_sorted['Timestamp'] < data_corte_com_gap)
].copy()
df_oot = df_sorted[df_sorted['Timestamp'] >= data_corte_com_gap].copy()

print("="*80)
print("DIVISÃO TEMPORAL REALIZADA COM GAP DE SEGURANÇA")
print("="*80)

print(f"\n📊 Dataset de Treino (df_treino):")
print(f"   - Total de registros: {len(df_treino):,}")
print(f"   - Período: {df_treino['Timestamp'].min()} até {df_treino['Timestamp'].max()}")
print(f"   - Casos de lavagem: {df_treino['Is Laundering'].sum():,}")
print(f"   - Taxa de lavagem: {(df_treino['Is Laundering'].sum() / len(df_treino) * 100):.2f}%")

print(f"\n⚠️  GAP de Segurança (descartado):")
print(f"   - Total de registros: {len(df_gap):,}")
print(f"   - Período: {df_gap['Timestamp'].min()} até {df_gap['Timestamp'].max()}")
print(f"   - Justificativa: Evita label leakage por chargeback delay")

print(f"\n📊 Dataset Out-of-Time (df_oot):")
print(f"   - Total de registros: {len(df_oot):,}")
print(f"   - Período: {df_oot['Timestamp'].min()} até {df_oot['Timestamp'].max()}")
print(f"   - Casos de lavagem: {df_oot['Is Laundering'].sum():,}")
print(f"   - Taxa de lavagem: {(df_oot['Is Laundering'].sum() / len(df_oot) * 100):.2f}%")

print(f"\n📅 Datas de Corte:")
print(f"   - Fim do Treino: {df_treino['Timestamp'].max()}")
print(f"   - Gap: {GAP_DAYS} dias")
print(f"   - Início do OOT: {df_oot['Timestamp'].min()}")
print(f"\n✅ Gap temporal implementado com sucesso!")

DIVISÃO TEMPORAL REALIZADA COM GAP DE SEGURANÇA

📊 Dataset de Treino (df_treino):
   - Total de registros: 14,375,186
   - Período: 2022-08-01 00:00:00 até 2022-10-18 00:52:00
   - Casos de lavagem: 15,527
   - Taxa de lavagem: 0.11%

⚠️  GAP de Segurança (descartado):
   - Total de registros: 1,197,966
   - Período: 2022-10-18 00:53:00 até 2022-10-25 00:52:00
   - Justificativa: Evita label leakage por chargeback delay

📊 Dataset Out-of-Time (df_oot):
   - Total de registros: 2,395,950
   - Período: 2022-10-25 00:53:00 até 2023-01-09 12:27:00
   - Casos de lavagem: 5,489
   - Taxa de lavagem: 0.23%

📅 Datas de Corte:
   - Fim do Treino: 2022-10-18 00:52:00
   - Gap: 7 dias
   - Início do OOT: 2022-10-25 00:53:00

✅ Gap temporal implementado com sucesso!


## 6. Salvamento dos Datasets

In [6]:
# Definir os caminhos de saída
output_dir = r'C:\Users\win\Desktop\Projetos\TCC_v2\anti_money_laundering\money_laundering\data\processed'
os.makedirs(output_dir, exist_ok=True)

# Caminhos dos arquivos de saída
treino_path = os.path.join(output_dir, 'df_treino.csv')
oot_path = os.path.join(output_dir, 'df_oot.csv')

# Remover a coluna Year_Month antes de salvar (coluna auxiliar)
df_treino_save = df_treino.drop(columns=['Year_Month'], errors='ignore')
df_oot_save = df_oot.drop(columns=['Year_Month'], errors='ignore')

# Salvar os datasets
print("Salvando datasets...")
df_treino_save.to_csv(treino_path, index=False)
df_oot_save.to_csv(oot_path, index=False)

print(f"\n✅ Datasets salvos com sucesso!")
print(f"\n📁 Arquivos salvos em:")
print(f"   - Treino: {treino_path}")
print(f"   - OOT: {oot_path}")

# Verificar tamanhos dos arquivos
treino_size = os.path.getsize(treino_path) / (1024 * 1024)  # MB
oot_size = os.path.getsize(oot_path) / (1024 * 1024)  # MB

print(f"\n📏 Tamanho dos arquivos:")
print(f"   - Treino: {treino_size:.2f} MB")
print(f"   - OOT: {oot_size:.2f} MB")

Salvando datasets...

✅ Datasets salvos com sucesso!

📁 Arquivos salvos em:
   - Treino: C:\Users\win\Desktop\Projetos\TCC_v2\anti_money_laundering\money_laundering\data\processed\df_treino.csv
   - OOT: C:\Users\win\Desktop\Projetos\TCC_v2\anti_money_laundering\money_laundering\data\processed\df_oot.csv

📏 Tamanho dos arquivos:
   - Treino: 2780.81 MB
   - OOT: 463.86 MB


## 7. Verificação Final

Vamos verificar a consistência dos dados salvos.

In [7]:
# Recarregar os arquivos para verificação
df_treino_verificacao = pd.read_csv(treino_path)
df_oot_verificacao = pd.read_csv(oot_path)

print("="*60)
print("VERIFICAÇÃO DOS ARQUIVOS SALVOS")
print("="*60)

print(f"\n✓ Dataset de Treino carregado:")
print(f"  - Registros: {len(df_treino_verificacao):,}")
print(f"  - Colunas: {len(df_treino_verificacao.columns)}")
print(f"  - Formato: {df_treino_verificacao.shape}")

print(f"\n✓ Dataset Out-of-Time carregado:")
print(f"  - Registros: {len(df_oot_verificacao):,}")
print(f"  - Colunas: {len(df_oot_verificacao.columns)}")
print(f"  - Formato: {df_oot_verificacao.shape}")

print(f"\n✓ Total de registros: {len(df_treino_verificacao) + len(df_oot_verificacao):,}")
print(f"✓ Registros originais: {len(df):,}")
print(f"✓ Diferença: {len(df) - (len(df_treino_verificacao) + len(df_oot_verificacao))}")

print("\n" + "="*60)
print("✅ DIVISÃO TEMPORAL CONCLUÍDA COM SUCESSO!")
print("="*60)

VERIFICAÇÃO DOS ARQUIVOS SALVOS

✓ Dataset de Treino carregado:
  - Registros: 14,375,186
  - Colunas: 17
  - Formato: (14375186, 17)

✓ Dataset Out-of-Time carregado:
  - Registros: 2,395,950
  - Colunas: 17
  - Formato: (2395950, 17)

✓ Total de registros: 16,771,136
✓ Registros originais: 17,969,102
✓ Diferença: 1197966

✅ DIVISÃO TEMPORAL CONCLUÍDA COM SUCESSO!
